# CIFAR-10 Frozen SNGP Checkpoint Evaluation

Load a frozen-backbone SNGP checkpoint, inspect parameter counts, and evaluate test accuracy and NLL.

In [1]:
from pathlib import Path
import os
import sys

import torch
import yaml

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = REPO_ROOT.parent

sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

from src.data.cifar10 import get_cifar10_loaders
from src.models.frozen_sngp import FrozenBackboneSNGPClassifier, load_frozen_resnet_encoder
from src.models.sngp import laplace_predictive_probs

In [2]:
CONFIG_PATH = REPO_ROOT / "configs" / "cifar10_frozen_sngp.yaml"
CHECKPOINT_DIR = REPO_ROOT / "checkpoints_frozen_sngp"

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print("checkpoint dir:", CHECKPOINT_DIR)

device: cuda
checkpoint dir: /w/20252/wjcai/uq/manygp/checkpoints_frozen_sngp


In [3]:
def load_best_checkpoint(checkpoint_dir: Path, device: torch.device):
    checkpoint_paths = sorted(checkpoint_dir.glob("cifar10_frozen_sngp*.pt"))
    if not checkpoint_paths:
        raise FileNotFoundError(f"No frozen SNGP checkpoint files found in {checkpoint_dir}")

    best_checkpoint = None
    best_path = None
    best_score = float("inf")

    for checkpoint_path in checkpoint_paths:
        checkpoint = torch.load(checkpoint_path, map_location=device)
        score = checkpoint.get("test_nll")
        if score is None:
            continue
        if score < best_score:
            best_score = score
            best_checkpoint = checkpoint
            best_path = checkpoint_path

    if best_checkpoint is None or best_path is None:
        raise ValueError(f"No frozen SNGP checkpoints with 'test_nll' found in {checkpoint_dir}")

    return best_path, best_checkpoint


BEST_CHECKPOINT_PATH, checkpoint = load_best_checkpoint(CHECKPOINT_DIR, device)
print("best checkpoint:", BEST_CHECKPOINT_PATH)
print("stored test nll:", checkpoint.get("test_nll"))
print("stored test accuracy:", checkpoint.get("test_accuracy"))

best checkpoint: /w/20252/wjcai/uq/manygp/checkpoints_frozen_sngp/cifar10_frozen_sngp_epoch085_nll0.3255.pt
stored test nll: 0.32549798069000246
stored test accuracy: 0.9195


In [7]:
# print(checkpoint)

In [4]:
backbone_cfg = cfg["backbone"]
frozen_encoder = load_frozen_resnet_encoder(
    checkpoint_path=backbone_cfg["checkpoint_path"],
    width=backbone_cfg["width"],
    embedding_dim=backbone_cfg["embedding_dim"],
    device=device,
)

model_cfg = cfg["model"]
model = FrozenBackboneSNGPClassifier(
    encoder=frozen_encoder,
    num_classes=model_cfg["num_classes"],
    hidden_dims=model_cfg["hidden_dims"],
    spec_norm_bound=model_cfg["spec_norm_bound"],
    dropout_rate=model_cfg["dropout_rate"],
    num_inducing=model_cfg["num_inducing"],
    ridge_penalty=model_cfg["ridge_penalty"],
    feature_scale=model_cfg["feature_scale"],
    gp_cov_momentum=model_cfg["gp_cov_momentum"],
    normalize_input=model_cfg["normalize_input"],
    kernel_type=model_cfg["kernel_type"],
    input_normalization=model_cfg["input_normalization"],
    kernel_scale=model_cfg["kernel_scale"],
    length_scale=model_cfg["length_scale"],
).to(device)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("loaded epoch:", checkpoint.get("epoch"))

loaded epoch: 85


In [8]:
print(model)

FrozenBackboneSNGPClassifier(
  (encoder): CifarResNetEncoder(
    (stem): Sequential(
      (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
    )
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (shortcut): Identity()
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): 

In [5]:
def count_parameters(module):
    total = sum(p.numel() for p in module.parameters())
    trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
    frozen = total - trainable
    return total, trainable, frozen


total_params, trainable_params, frozen_params = count_parameters(model)
encoder_total, encoder_trainable, encoder_frozen = count_parameters(model.encoder)
head_total, head_trainable, head_frozen = count_parameters(model.head)
gp_total, gp_trainable, gp_frozen = count_parameters(model.classifier)

print(f"Model total params:     {total_params:,}")
print(f"Model trainable params: {trainable_params:,}")
print(f"Model frozen params:    {frozen_params:,}")
print()
print(f"Encoder total params:   {encoder_total:,}")
print(f"Head total params:      {head_total:,}")
print(f"GP head total params:   {gp_total:,}")

Model total params:     11,261,258
Model trainable params: 26,762
Model frozen params:    11,234,496

Encoder total params:   11,234,496
Head total params:      16,512
GP head total params:   10,250


In [ ]:
data_cfg = cfg["data"]
_, test_loader, _, test_dataset = get_cifar10_loaders(
    data_root=data_cfg["root"],
    batch_size=data_cfg["batch_size"],
    num_workers=0,
    smoke_test=False,
)
print("test size:", len(test_dataset))

In [ ]:
@torch.no_grad()
def evaluate_frozen_sngp(model, loader, device, num_mc_samples=10):
    model.eval()
    total_correct = 0
    total_examples = 0
    total_nll = 0.0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        logits, variances = model(images, return_cov=True)
        probs = laplace_predictive_probs(logits, variances, num_mc_samples=num_mc_samples)
        log_probs = probs.clamp_min(1e-12).log()

        total_correct += (probs.argmax(dim=1) == labels).sum().item()
        total_examples += labels.size(0)
        total_nll += -log_probs.gather(1, labels.unsqueeze(1)).sum().item()

    return {
        "accuracy": total_correct / total_examples,
        "nll": total_nll / total_examples,
    }


metrics = evaluate_frozen_sngp(
    model=model,
    loader=test_loader,
    device=device,
    num_mc_samples=cfg["training"].get("num_mc_samples", 10),
)
print(f"Test accuracy: {metrics['accuracy'] * 100:.2f}%")
print(f"Test NLL: {metrics['nll']:.4f}")